In [ ]:
#pip install pbix-mcp postgresql-server-mcp 

#powerbi-modeling-mcp

In [ ]:
import os
import sys
import json
import httpx
from dotenv import load_dotenv

# 1. Подключаем ваш файл с кредами PostgreSQL
sys.path.append(r'C:\Users\maxim\Desktop\PowerBiAgent')
from conn import pg

# 2. Загружаем API-ключ GLM из .env
load_dotenv()
API_KEY = os.getenv("GLM_API_KEY")
if not API_KEY:
    raise ValueError("Ключ GLM_API_KEY не найден в .env")

BASE_URL = "https://agent-api.bigmodel.cn/api/agent/managed"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "zai-version": "2026-05-26",
    "zai-beta": "managed-agents-2026-09-22"
}

# 3. Собираем строку подключения к PostgreSQL из conn.py
DB_URL = f"postgresql://{pg['user']}:{pg['password']}@{pg['host']}:{pg['port']}/{pg['dbname']}"

# ==========================================
# СОЗДАНИЕ ОПТИМИЗАТОРА
# ==========================================
print("Создаем агента-оптимизатора...")
optimizer_payload = {
    "name": "model-optimizer",
    "model": "glm-5.3",
    "system": "Ты — эксперт по DAX и моделированию. Анализируй существующие модели Power BI, оптимизируй DAX-меры и связи. Используй MCP-сервер 'powerbi-modeling'.",
    "tools": [{"type": "agent_toolset_20260601"}],
    "mcp_servers": {
        "powerbi-modeling": {
            "command": "npx",
            "args": ["-y", "@microsoft/powerbi-modeling-mcp", "--start", "--accept-eula"]
        }
    }
}
opt_resp = httpx.post(f"{BASE_URL}/v1/agents", headers=HEADERS, json=optimizer_payload)
opt_resp.raise_for_status()
optimizer_id = opt_resp.json()["id"]
print(f"✅ Оптимизатор создан: {optimizer_id}")

# ==========================================
# СОЗДАНИЕ КООРДИНАТОРА
# ==========================================
print("\nСоздаем агента-координатора...")
coordinator_payload = {
    "name": "dashboard-creator",
    "model": "glm-5.3",
    "system": "Ты — архитектор дашбордов. Создавай .pbix через MCP 'pbix-creator', читай данные из PostgreSQL через 'postgres'. Для сложных DAX-мер делегируй задачу 'model-optimizer'.",
    "tools": [{"type": "agent_toolset_20260601"}],
    "mcp_servers": {
        "pbix-creator": {
            "command": "python",
            "args": ["-m", "pbix_mcp.cli"]
        },
        "postgres": {
            "command": "uvx",
            "args": ["mcpg"],
            "env": {"MCPG_DATABASE_URL": DB_URL}
        }
    },
    "multiagent": {
        "type": "coordinator",
        "agents": [
            {"type": "agent", "id": optimizer_id},
            {"type": "self"}
        ]
    }
}
coord_resp = httpx.post(f"{BASE_URL}/v1/agents", headers=HEADERS, json=coordinator_payload)
coord_resp.raise_for_status()
coordinator_id = coord_resp.json()["id"]
print(f"✅ Координатор создан: {coordinator_id}")

# ==========================================
# СОХРАНЕНИЕ ID В ФАЙЛ
# ==========================================
ids_to_save = {
    "optimizer_id": optimizer_id,
    "coordinator_id": coordinator_id
}
with open("agent_ids.json", "w", encoding="utf-8") as f:
    json.dump(ids_to_save, f, indent=2)

print(f"\n💾 ID сохранены в agent_ids.json")
print(f"   Координатор: {coordinator_id}")